# Sociodemographic Predictors of Hypertension Risk Among Nigerian Adults
## A Machine Learning Analysis of Nigeria DHS 2024

**Author:** Anthony Okeibuno | Health Data Systems Strategist  
**Data:** Nigeria Demographic and Health Survey (NDHS) 2024 — DHS-VIII  
**Last updated:** April 2026

---

### What this notebook does

Hypertension is the leading modifiable risk factor for cardiovascular disease in sub-Saharan Africa. In Nigeria, a large share of hypertensive adults have never been screened. Meaning they carry the condition silently, untreated.

This notebook builds a machine learning model that answers one question:  
**Can we identify which unscreened Nigerian adults are most likely to be hypertensive, using only sociodemographic information we already know about them?**

The approach:
1. Among adults who *have* been screened, train a model to predict hypertension diagnosis from sociodemographic features
2. Apply that model to adults who have *never* been screened to estimate their risk
3. Disaggregate results by geopolitical zone to identify where screening programs should be prioritised

### Notebook structure
1. Data Loading
2. Variable Discovery
3. Target Variable Analysis
4. Feature Selection
5. Dataset Construction
6. Modelling
7. SHAP Interpretation
8. Zonal Analysis
9. Unscreened Population Scoring

---
## 1. Data Loading

The DHS Program distributes data in Stata (.dta) format, which embeds variable labels and value labels directly in the file. We use `pyreadstat` to read it into pandas while preserving those labels. This saves a lot of dictionary-hunting later.

Two files are needed:
- **NGIR8BFL.DTA** - Individual Recode (women 15–49), 39,050 rows
- **NGMR8BFL.DTA** - Men's Recode (men 15–59), 12,204 rows

Both require a registered DHS account to download: dhsprogram.com

In [ ]:
import pyreadstat
import pandas as pd
import numpy as np

# apply_value_formats=False keeps variables as numeric codes (0, 1, 2...)
# rather than string labels. Numeric codes are cleaner for modelling.
ir, ir_meta = pyreadstat.read_dta("../data/raw/NGIR8BFL.DTA", apply_value_formats=False)
mr, mr_meta = pyreadstat.read_dta("../data/raw/NGMR8BFL.DTA", apply_value_formats=False)

print("IR shape:", ir.shape)   # (rows, columns)
print("MR shape:", mr.shape)

---
## 2. Variable Discovery

The DHS-VIII questionnaire includes a new set of chronic disease variables (CHD01–CHD10) added specifically in the 2024 round. These were not present in earlier Nigeria DHS surveys, making this dataset uniquely suited for NCD analysis.

We search the metadata — the embedded variable labels — to find relevant variables rather than manually scanning a 6,000-column dataset.

In [ ]:
# Convert the metadata label dictionary to a pandas Series so we can search it
ir_vars = pd.Series(ir_meta.column_names_to_labels)
mr_vars = pd.Series(mr_meta.column_names_to_labels)

# Search for chronic disease variables (women)
print("=== Women's CHD variables ===")
print(ir_vars[ir_vars.index.str.contains("chd", case=False)])

print("\n=== Men's CHD variables ===")
print(mr_vars[mr_vars.index.str.contains("chd", case=False)])

# Also check for any directly measured blood pressure variables
# (these exist in some DHS surveys as biomarker data)
print("\n=== Blood pressure / hypertension variables (women) ===")
print(ir_vars[ir_vars.index.str.contains("bp|blood|press|hypert", case=False)])

---
## 3. Target Variable Analysis

The CHD variables give us a clean two-stage screening picture:

| Variable | Question |
|----------|----------|
| chd01 | Ever had blood pressure measured by a doctor/healthcare worker? |
| chd02 | Ever been told you have high blood pressure / hypertension? |
| chd03 | Told in the past 12 months? |
| chd04 | Prescribed medication to control it? |
| chd05 | Currently taking that medication? |

**Our target variable is chd02**: hypertension diagnosis — applied only to those who confirmed they were screened (chd01 = 1). The unscreened group (chd01 = 0) becomes the inference population at the end.

In [ ]:
# Check value distributions for women
print("=== Women ===")
for col in ["chd01", "chd02", "chd03", "chd04", "chd05"]:
    print(f"\n{col} -- {ir_meta.column_names_to_labels[col]}")
    print(ir[col].value_counts(dropna=False))

# Check value distributions for men
print("\n=== Men ===")
for col in ["mchd01", "mchd02", "mchd03", "mchd04", "mchd05"]:
    print(f"\n{col} -- {mr_meta.column_names_to_labels[col]}")
    print(mr[col].value_counts(dropna=False))

**Key observations from the distributions:**

- Women screened: 20,036 | Diagnosed hypertensive: 2,975 (~14.8% of screened)
- Men screened: 4,297 | Diagnosed hypertensive: 754 (~17.6% of screened)
- Never screened: 18,951 women + 7,722 men = **26,673 adults** — our inference population
- chd01 = 8 means "don't know" — we'll exclude these 248 respondents

A ~15% positive rate is workable. No aggressive class imbalance handling needed.

---
## 4. Feature Selection

We restrict to sociodemographic features available for both men and women across all zones. The goal is a model that requires no clinical measurements — only information already collected in the survey.

Candidates checked and decisions made:
- **v445 (BMI)**: 64% missing — only collected in biomarker-selected households. Excluded to avoid selection bias.
- **v701 (partner's education)**: 34% missing — only applies to partnered women. Excluded.
- **mv467d (distance to facility, men)**: variable does not exist in men's file. Excluded from both.
- **v463a (smokes cigarettes)**: 98.5% zero for women, all missing for men. Near-zero variance. Excluded.

In [ ]:
# Check candidate variables for women
feature_candidates = [
    'v012',   # age
    'v025',   # urban/rural
    'v106',   # education level (categorical)
    'v190',   # wealth index (1=poorest, 5=richest)
    'v133',   # years of education (continuous)
    'v463a',  # smokes cigarettes
    'v445',   # BMI
    'szone',  # geopolitical zone
    'v701',   # partner's education
    'v731',   # worked in last 12 months
    'v157',   # reads newspaper
    'v151',   # sex of household head
    'v201',   # total children ever born (parity)
]

for var in feature_candidates:
    if var in ir.columns:
        label = ir_meta.column_names_to_labels.get(var, 'no label')
        missing_pct = ir[var].isna().mean() * 100
        print(f"{var}: {label} | Missing: {missing_pct:.1f}%")
    else:
        print(f"{var}: NOT FOUND")

---
## 5. Dataset Construction

We merge the women's and men's records into a single dataset with a `sex` indicator column. Men's variables follow the same naming convention but are prefixed with `mv` — we rename them to match women's columns before combining.

In [ ]:
# ── Women's features ──────────────────────────────────────────────────────────
ir_features = ir[[
    'chd01', 'chd02',
    'v012',   # age
    'v025',   # urban/rural
    'v106',   # education level
    'v190',   # wealth index
    'v133',   # years of education
    'szone',  # geopolitical zone
    'v731',   # worked in last 12 months
    'v157',   # reads newspaper/magazine
    'v151',   # sex of household head
    'v201',   # total children ever born
]].copy()
ir_features['sex'] = 0  # 0 = female

# ── Men's features ────────────────────────────────────────────────────────────
# Men's variables use 'mv' prefix and 'smzone' instead of 'szone'
mr_features = mr[[
    'mchd01', 'mchd02',
    'mv012', 'mv025', 'mv106', 'mv190', 'mv133',
    'smzone', 'mv731', 'mv157', 'mv151', 'mv201',
]].copy()

# Rename men's columns to match women's — required for concat to align correctly
mr_features.columns = [
    'chd01', 'chd02',
    'v012', 'v025', 'v106', 'v190', 'v133',
    'szone', 'v731', 'v157', 'v151', 'v201',
]
mr_features['sex'] = 1  # 1 = male

# ── Combine into one dataset ──────────────────────────────────────────────────
df = pd.concat([ir_features, mr_features], ignore_index=True)

# ── Exclude "don't know" screening responses (chd01 = 8) ─────────────────────
# These 248 respondents can't be reliably placed in screened or unscreened groups
df = df[df['chd01'] != 8].copy()

# ── Split: screened vs unscreened ─────────────────────────────────────────────
# Screened (chd01 = 1): model is trained on this group
# Unscreened (chd01 = 0): model is applied to this group at the end
screened   = df[df['chd01'] == 1].copy()
unscreened = df[df['chd01'] == 0].copy()

# ── Define target variable ────────────────────────────────────────────────────
# chd02 = 1 means the respondent was told they have hypertension
screened['hypertensive'] = screened['chd02'].astype(int)

print("Combined shape:", df.shape)
print("\nScreened population:", screened.shape)
print("Target distribution:")
print(screened['hypertensive'].value_counts())
print(f"Positive rate: {screened['hypertensive'].mean():.1%}")
print("\nUnscreened (inference target later):", unscreened.shape)
print("\nMissing values in screened set:")
missing = screened.isnull().sum()
print(missing[missing > 0] if missing[missing > 0].any() else "None")

In [ ]:
# ── Final feature list ────────────────────────────────────────────────────────
# These are the 11 variables the model will be trained on
features = [
    'v012',  # age
    'v025',  # urban/rural residence
    'v106',  # education level (categorical: 0=none, 1=primary, 2=secondary, 3=higher)
    'v190',  # wealth index (1=poorest to 5=richest)
    'v133',  # years of education (continuous)
    'szone', # geopolitical zone (1=NC, 2=NE, 3=NW, 4=SE, 5=SS, 6=SW)
    'v731',  # worked in last 12 months
    'v157',  # reads newspaper/magazine
    'v151',  # sex of household head
    'v201',  # total children ever born (parity)
    'sex',   # 0=female, 1=male
]

X = screened[features].copy()
y = screened['hypertensive'].copy()

# Save the unscreened feature matrix for later inference
unscreened_df = unscreened.copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nAny missing values:", X.isnull().sum().sum())
print("\nFeature dtypes:")
print(X.dtypes)

---
## 6. Modelling

We compare three models that represent different learning strategies:
- **Logistic Regression**: linear baseline, interpretable coefficients
- **Random Forest**: ensemble of decision trees, handles non-linearity
- **Gradient Boosting**: sequential tree ensemble, often strongest on tabular data

Evaluation metric: **AUC-ROC** (Area Under the ROC Curve)
- AUC = 0.5 means the model is no better than random guessing
- AUC = 1.0 means perfect discrimination
- For a population screening tool, AUC is more informative than accuracy because it evaluates performance across all decision thresholds

We use **stratified 5-fold cross-validation** on the training set to avoid overfitting to a single split, then confirm on a held-out test set.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# ── Train/test split ──────────────────────────────────────────────────────────
# stratify=y ensures both splits have the same 13.7% positive rate
# random_state=42 makes results reproducible
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Train positive rate: {y_train.mean():.1%}")
print(f"Test positive rate:  {y_test.mean():.1%}")

# ── Define models ─────────────────────────────────────────────────────────────
# Logistic Regression needs StandardScaler because it's sensitive to feature scale
# Tree-based models (RF, GB) are scale-invariant — no scaling needed
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,   # 200 trees — good balance of performance vs. compute
        random_state=42,
        n_jobs=-1           # use all CPU cores
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,   # 200 boosting rounds
        random_state=42
    ),
}

# ── 5-Fold cross-validated AUC on training set ────────────────────────────────
# This gives us the model's expected performance on unseen data
# ± shows the variance across folds — lower is more stable
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n── 5-Fold CV AUC (train set) ──")
for name, model in models.items():
    scores = cross_val_score(
        model, X_train, y_train,
        cv=cv, scoring="roc_auc", n_jobs=-1
    )
    print(f"{name:25s}  AUC: {scores.mean():.4f} ± {scores.std():.4f}")

# ── Fit all models on the full training set ───────────────────────────────────
for name, model in models.items():
    model.fit(X_train, y_train)

# ── Evaluate on the held-out test set ────────────────────────────────────────
# Test set was never seen during training or CV — this is our honest final score
print("\n── Test Set AUC ──")
fig, ax = plt.subplots(figsize=(7, 5))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

for (name, model), color in zip(models.items(), colors):
    # predict_proba returns [P(class=0), P(class=1)] for each row
    # we take [:, 1] to get the probability of being hypertensive
    y_prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    print(f"{name:25s}  AUC: {auc:.4f}")

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, color=color, linewidth=1.8,
            label=f"{name}  (AUC = {auc:.3f})")

# Diagonal reference line — represents a model with no discriminative ability
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random chance")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves -- Hypertension Risk Model (Nigeria DHS 2024)")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig("../outputs/roc_curves.png", dpi=150)
plt.show()
print("Saved: outputs/roc_curves.png")

**Model selection: Gradient Boosting (AUC = 0.669)**

Gradient Boosting outperforms both alternatives. Random Forest underperforms (AUC 0.60) — likely because it builds trees independently and struggles with the low signal-to-noise ratio in this dataset. Gradient Boosting corrects errors iteratively, making it better suited here.

AUC of 0.67 using *only* sociodemographic features — no blood pressure readings, no BMI, no family history — is consistent with comparable population-level screening models in the literature. The framing matters: this is a surveillance tool, not a clinical diagnostic.

In [ ]:
# Select the final model for all downstream analysis
final_model = models["Gradient Boosting"]

---
## 7. SHAP Interpretation

SHAP (SHapley Additive exPlanations) assigns each feature a contribution value for each individual prediction. It answers: *for this specific person, how much did each feature push the model toward or away from a hypertension prediction?*

- **Positive SHAP value** = pushes toward hypertensive prediction
- **Negative SHAP value** = pushes toward not-hypertensive prediction
- **Beeswarm plot**: each dot is one person. X-axis = SHAP value. Color = feature value (red = high, blue = low)
- **Bar plot**: average absolute SHAP — overall feature importance

We use `TreeExplainer`, which is optimised for tree-based models and computes exact (not approximate) Shapley values.

In [ ]:
import shap

# Human-readable names to replace DHS variable codes in all plots
feature_names = [
    "Age",
    "Urban/Rural Residence",
    "Education Level",
    "Wealth Index",
    "Years of Education",
    "Geopolitical Zone",
    "Worked in Last 12 Months",
    "Reads Newspaper",
    "Sex of Household Head",
    "Total Children Ever Born",
    "Sex (Male=1)",
]

# Apply readable names to the test set
X_test_named = X_test.copy()
X_test_named.columns = feature_names

# ── Build explainer and compute SHAP values ───────────────────────────────────
# TreeExplainer is specific to tree-based models — faster and exact
explainer = shap.TreeExplainer(final_model)

# shap_values shape: (n_samples, n_features)
# Each value represents a feature's contribution to one prediction
shap_values = explainer.shap_values(X_test_named)

# ── Plot 1: Beeswarm summary ──────────────────────────────────────────────────
# Best plot for understanding HOW each feature affects predictions
# (direction + magnitude + distribution across the population)
plt.figure()
shap.summary_plot(
    shap_values,
    X_test_named,
    feature_names=feature_names,
    show=False
)
plt.title("SHAP Summary -- Hypertension Risk Drivers (Nigeria DHS 2024)")
plt.tight_layout()
plt.savefig("../outputs/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/shap_summary.png")

# ── Plot 2: Mean absolute SHAP (bar chart) ────────────────────────────────────
# Best plot for communicating overall feature importance to non-technical audiences
plt.figure()
shap.summary_plot(
    shap_values,
    X_test_named,
    feature_names=feature_names,
    plot_type="bar",
    show=False
)
plt.title("Mean |SHAP| -- Feature Importance (Nigeria DHS 2024)")
plt.tight_layout()
plt.savefig("../outputs/shap_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/shap_importance.png")

**SHAP findings:**

1. **Age** dominates — SHAP values extend to +2.5 for the oldest respondents. The direction is intuitive (older = higher risk), but the magnitude relative to all other features is striking.

2. **Geopolitical Zone** is the second strongest predictor — more important than wealth, education, or urban/rural status. This is the project's most policy-relevant finding. Geography within Nigeria carries structural meaning: dietary patterns, healthcare access density, and conflict exposure all vary by zone and are not captured by any other variable in the model.

3. **Total Children Ever Born** ranks third — higher parity is associated with higher risk, likely reflecting cumulative cardiovascular and metabolic stress across pregnancies, independent of age.

4. **Wealth Index, Years of Education, Reads Newspaper** show modest effects, mostly near zero. The low importance of these access-to-information proxies suggests that screening behaviour in Nigeria is driven more by structural barriers than individual awareness.

---
## 8. Zonal Analysis

SHAP tells us geopolitical zone matters. This section quantifies exactly how — computing diagnosed hypertension rates among screened adults in each of Nigeria's six geopolitical zones and comparing against the national average.

In [ ]:
# Zone code mapping: szone values from DHS Nigeria documentation
zone_map = {
    1: "North Central",
    2: "North East",
    3: "North West",
    4: "South East",
    5: "South South",
    6: "South West"
}

# Attach readable zone labels to the screened dataset
screened['zone_label'] = screened['szone'].map(zone_map)

# ── Hypertension prevalence by zone ──────────────────────────────────────────
zone_stats = (
    screened.groupby('zone_label')['hypertensive']
    .agg(
        diagnosed_count='sum',   # number diagnosed hypertensive in this zone
        total_screened='count',  # total screened in this zone
        prevalence='mean'        # proportion diagnosed (0 to 1)
    )
    .reset_index()
    .sort_values('prevalence', ascending=False)
)

zone_stats['prevalence_pct'] = (zone_stats['prevalence'] * 100).round(1)
print(zone_stats[['zone_label', 'diagnosed_count', 'total_screened', 'prevalence_pct']])

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

# Colour code: red = above national average, blue = below
national_avg = screened['hypertensive'].mean()
colors = ['#c0392b' if p > national_avg else '#2980b9'
          for p in zone_stats['prevalence']]

ax.barh(zone_stats['zone_label'], zone_stats['prevalence_pct'], color=colors)

# National average reference line
ax.axvline(national_avg * 100, color='black', linestyle='--',
           linewidth=1.2, label=f'National avg: {national_avg*100:.1f}%')

# Add percentage labels on each bar
for i, val in enumerate(zone_stats['prevalence_pct']):
    ax.text(val + 0.2, i, f'{val}%', va='center', fontsize=10)

ax.set_xlabel("Diagnosed Hypertension Rate (%)")
ax.set_title("Hypertension Prevalence by Geopolitical Zone\n"
             "Among Screened Adults -- Nigeria DHS 2024")
ax.legend()
plt.tight_layout()
plt.savefig("../outputs/zone_prevalence.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/zone_prevalence.png")

**A clear North-South gradient emerges:**

- North East: 16.7% | North Central: 14.8% | North West: 14.4% — all above national average (13.7%)
- South South: 14.2% — marginally above
- South West: 11.1% | South East: 10.4% — both below

The 6.3 percentage point gap between North East and South East is not noise — it reflects structural differences in diet (salt consumption patterns), health facility density, age profiles, and in the North East specifically, the impact of conflict-related displacement on healthcare access and health outcomes.

South West's relatively lower rate despite being the most urbanised zone likely reflects younger respondent age profiles (Lagos-dominated sample) and higher health literacy.

---
## 9. Unscreened Population Scoring

This is the public health payload of the project. We apply the trained model to the 26,673 adults who have never had their blood pressure checked — generating a predicted hypertension risk score for each person.

We then summarise by zone: what proportion of each zone's unscreened adults are estimated to be at high risk (predicted probability > 15%, slightly above the national diagnosed prevalence of 13.7%)?

In [ ]:
# ── Apply model to unscreened population ──────────────────────────────────────
# The model was trained on the same feature columns — use original variable names
X_unscreened = unscreened_df[features].copy()

# predict_proba[:, 1] gives the estimated probability of being hypertensive
# for each unscreened adult
unscreened_df['predicted_risk'] = final_model.predict_proba(X_unscreened)[:, 1]
unscreened_df['zone_label'] = unscreened_df['szone'].map(zone_map)

# ── Summarise estimated risk by zone ─────────────────────────────────────────
# risk_threshold = 0.15 is set slightly above the national prevalence of 13.7%
# Adults above this threshold are flagged as high-priority for outreach
risk_threshold = 0.15

zone_risk = (
    unscreened_df.groupby('zone_label')
    .agg(
        unscreened_count=('predicted_risk', 'count'),
        mean_predicted_risk=('predicted_risk', 'mean'),
        high_risk_count=('predicted_risk', lambda x: (x > risk_threshold).sum())
    )
    .reset_index()
)

# Convert proportions to percentages for readability
zone_risk['high_risk_pct'] = (
    zone_risk['high_risk_count'] / zone_risk['unscreened_count'] * 100
).round(1)

zone_risk['mean_predicted_risk'] = (
    zone_risk['mean_predicted_risk'] * 100
).round(1)

zone_risk = zone_risk.sort_values('mean_predicted_risk', ascending=False)

print("── Estimated Hypertension Risk Among UNSCREENED Adults ──")
print(zone_risk.to_string(index=False))

# ── Save individual-level risk scores ────────────────────────────────────────
# This CSV can be used by health programs to segment and prioritise outreach
unscreened_df[['szone', 'zone_label', 'sex', 'v012',
               'v190', 'v106', 'predicted_risk']].to_csv(
    "../outputs/unscreened_risk_scores.csv", index=False
)
print("\nSaved: outputs/unscreened_risk_scores.csv")

**The North-South pattern sharpens when we look at unscreened adults:**

The North East has the highest *diagnosed* rate among screened adults (16.7%) AND the highest *estimated* risk among unscreened adults (12.8% mean, 32.3% above the high-risk threshold). This convergence — two independent analyses pointing at the same zone — is what makes the finding credible rather than an artefact.

**Headline finding: 1 in 3 unscreened adults in the North East is estimated to be at high hypertension risk and has never had their blood pressure checked.**

This is the number that translates directly into a policy recommendation: prioritise blood pressure screening outreach in North East Nigeria, particularly among older adults and those with high parity.